In [ ]:
## Extract the NBA Teams for the current season into a DataFrame

import requests
import pandas as pd
import time
from datetime import datetime

BASE = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba"

def extract_teams():
    url = f"{BASE}/teams"
    data = requests.get(url).json()
    # Looking into the data we extracted to find what we're looking for

    print("Checking keys for data: ", data.keys()) 
    print("Checking keys for data['sports']: ", data['sports'][0].keys())
    print("Checking keys for data['sports'][0]['leagues'][0]: ", data["sports"][0]["leagues"][0].keys())

    teams = []
    for team_obj in data["sports"][0]["leagues"][0]["teams"]:
        team = team_obj["team"]

        teams.append({
            "team_id": team["id"],
            "name": team["displayName"],
            "abbreviation": team["abbreviation"],
            "location": team["location"],
            "logo": team["logos"][0]["href"] if team.get("logos") else None
        })

    return pd.DataFrame(teams)

df_teams = extract_teams()

df_teams.head()

print("Total Active Teams:", len(df_teams))

Checking keys for data:  dict_keys(['sports'])
Checking keys for data['sports']:  dict_keys(['id', 'uid', 'name', 'slug', 'leagues'])
Checking keys for data['sports'][0]['leagues'][0]:  dict_keys(['id', 'uid', 'name', 'abbreviation', 'shortName', 'slug', 'teams', 'year', 'season'])
Total Active Teams: 30


In [ ]:
df_teams

,team_id,name,abbreviation,location,logo
0,1,Atlanta Hawks,ATL,Atlanta,https://a.espncdn.com/i/teamlogos/nba/500/atl.png
1,2,Boston Celtics,BOS,Boston,https://a.espncdn.com/i/teamlogos/nba/500/bos.png
2,17,Brooklyn Nets,BKN,Brooklyn,https://a.espncdn.com/i/teamlogos/nba/500/bkn.png
3,30,Charlotte Hornets,CHA,Charlotte,https://a.espncdn.com/i/teamlogos/nba/500/cha.png
4,4,Chicago Bulls,CHI,Chicago,https://a.espncdn.com/i/teamlogos/nba/500/chi.png
5,5,Cleveland Cavaliers,CLE,Cleveland,https://a.espncdn.com/i/teamlogos/nba/500/cle.png
6,6,Dallas Mavericks,DAL,Dallas,https://a.espncdn.com/i/teamlogos/nba/500/dal.png
7,7,Denver Nuggets,DEN,Denver,https://a.espncdn.com/i/teamlogos/nba/500/den.png
8,8,Detroit Pistons,DET,Detroit,https://a.espncdn.com/i/teamlogos/nba/500/det.png
9,9,Golden State Warriors,GS,Golden State,https://a.espncdn.com/i/teamlogos/nba/500/gs.png


In [ ]:
## Extract Active Players in the NBA and place in a DataFrame

def extract_active_players(df_teams):
    players = []

    for team_id in df_teams["team_id"]:
        url = f"{BASE}/teams/{team_id}/roster" # https://sportsapis.dev/espn-api#endpoints
        
        roster = requests.get(url).json() # dict_keys(['timestamp', 'status', 'season', 'athletes', 'coach', 'team'])
       
        # print(roster.keys())
        #print(roster['athletes'][0].keys())

        for athlete in roster.get("athletes", []): 
    # dict_keys(['id', 'uid', 'guid', 'alternateIds', 'firstName', 'lastName', 'fullName', 'displayName', 'shortName', 
    # 'weight', 'displayWeight', 'height', 'displayHeight', 'age', 'dateOfBirth', 'debutYear', 'links', 'birthPlace', 'college', 
    # 'slug', 'headshot', 'jersey', 'position', 'injuries', 'teams', 'contracts', 'experience', 'contract', 'status'])

            players.append({
                "player_id": athlete["id"],
                "full_name": athlete["fullName"],
                "position": athlete.get("position", {}).get("abbreviation"),
                "jersey": athlete.get("jersey"),
                "height": athlete.get("height"),
                "weight": athlete.get("weight"),
                "age": athlete.get("age"),
                "team_id": team_id,
                "active": athlete.get("active", True)
            })

        time.sleep(0.3)  # polite delay

    return pd.DataFrame(players)

df_players = extract_active_players(df_teams)
print(df_players.head())
print("Total Active Players:", len(df_players))


dict_keys(['id', 'uid', 'guid', 'alternateIds', 'firstName', 'lastName', 'fullName', 'displayName', 'shortName', 'weight', 'displayWeight', 'height', 'displayHeight', 'age', 'dateOfBirth', 'debutYear', 'links', 'birthPlace', 'college', 'slug', 'headshot', 'jersey', 'position', 'injuries', 'teams', 'contracts', 'experience', 'contract', 'status'])
dict_keys(['id', 'uid', 'guid', 'alternateIds', 'firstName', 'lastName', 'fullName', 'displayName', 'shortName', 'weight', 'displayWeight', 'height', 'displayHeight', 'age', 'dateOfBirth', 'links', 'birthPlace', 'college', 'slug', 'headshot', 'jersey', 'position', 'injuries', 'teams', 'contracts', 'experience', 'status'])
dict_keys(['id', 'uid', 'guid', 'alternateIds', 'firstName', 'lastName', 'fullName', 'displayName', 'shortName', 'weight', 'displayWeight', 'height', 'displayHeight', 'age', 'dateOfBirth', 'links', 'birthPlace', 'college', 'slug', 'headshot', 'position', 'injuries', 'teams', 'contracts', 'experience', 'contract', 'status'])
d

KeyboardInterrupt: 

In [13]:
len(df_players)

528

In [ ]:
## Extract Current Standings and Place in a DataFrame


import requests
import pandas as pd

STANDINGS_URL = "https://site.web.api.espn.com/apis/v2/sports/basketball/nba/standings?level=3"

def _find_entries_anywhere(obj):
    """Fallback: recursively find the first non-empty standings.entries list."""
    if isinstance(obj, dict):
        st = obj.get("standings")
        if isinstance(st, dict):
            entries = st.get("entries")
            if isinstance(entries, list) and entries:
                return entries
        for v in obj.values():
            found = _find_entries_anywhere(v)
            if found:
                return found
    elif isinstance(obj, list):
        for item in obj:
            found = _find_entries_anywhere(item)
            if found:
                return found
    return []

def extract_standings():
    r = requests.get(STANDINGS_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=30) 
    r.raise_for_status()
    data = r.json() # dict_keys(['uid', 'id', 'name', 'abbreviation', 'shortName', 'children', 'isConference', 'season', 'links', 'seasons'])
    print("checking keys for data: ", data['children'][0].keys())

    rows = []

    children = data.get("children", []) # dict_keys(['uid', 'id', 'name', 'abbreviation', 'isConference', 'standings'])
    # Expected: 2 conferences (Eastern, Western). If present, parse the tree.
    print("checking keys for children: ", children.keys())
    if children:
        for conf in children: # dict_keys(['id', 'name', 'displayName', 'links', 'season', 'seasonType', 'seasonDisplayName', 'entries'])
            conf_name = conf.get("name")
            for div in conf.get("children", []):
                div_name = div.get("name")
                for entry in div.get("standings", {}).get("entries", []): 
                    team = entry.get("team", {}) # dict_keys(['id', 'uid', 'location', 'name', 'abbreviation', 'displayName', 'shortDisplayName', 'isActive', 'logos', 'links'])
                    stats = {s.get("name"): s.get("value") for s in entry.get("stats", []) if isinstance(s, dict)}
                    rows.append({
                        "team_id": team.get("id"),
                        "team_name": team.get("displayName"),
                        "conference": conf_name,
                        "division": div_name,
                        "wins": stats.get("wins"),
                        "losses": stats.get("losses"),
                        "win_pct": stats.get("winPercent"),
                        "games_back": stats.get("gamesBack"),
                        "streak": stats.get("streak"),
                    })
    else:
        # Fallback parse if ESPN returns a flatter structure
        for entry in _find_entries_anywhere(data):
            team = entry.get("team", {})
            stats = {s.get("name"): s.get("value") for s in entry.get("stats", []) if isinstance(s, dict)}
            rows.append({
                "team_id": team.get("id"),
                "team_name": team.get("displayName"),
                "conference": team.get("conference", {}).get("name"),
                "division": team.get("division", {}).get("name") or stats.get("division"),
                "wins": stats.get("wins"),
                "losses": stats.get("losses"),
                "win_pct": stats.get("winPercent"),
                "games_back": stats.get("gamesBack"),
                "streak": stats.get("streak"),
            })

    df = pd.DataFrame(rows).dropna(subset=["team_id"]).drop_duplicates(subset=["team_id"])
    return df

df_standings = extract_standings()
print("team standings:", len(df_standings))
print(df_standings.sort_values(["conference", "wins"], ascending=[True, False]).head(10))




checking keys for data:  dict_keys(['uid', 'id', 'name', 'abbreviation', 'children', 'isConference'])


In [29]:
df_standings

,team_id,team_name,conference,division,wins,losses,win_pct,games_back,streak
0,17,Brooklyn Nets,Eastern Conference,Atlantic,15.0,42.0,0.263158,None,-5.0
1,20,Philadelphia 76ers,Eastern Conference,Atlantic,32.0,26.0,0.551724,None,2.0
2,28,Toronto Raptors,Eastern Conference,Atlantic,34.0,25.0,0.576271,None,-2.0
3,18,New York Knicks,Eastern Conference,Atlantic,37.0,22.0,0.627119,None,-1.0
4,2,Boston Celtics,Eastern Conference,Atlantic,38.0,20.0,0.655172,None,-1.0
5,11,Indiana Pacers,Eastern Conference,Central,15.0,44.0,0.254237,None,-4.0
6,4,Chicago Bulls,Eastern Conference,Central,24.0,35.0,0.406780,None,-10.0
7,15,Milwaukee Bucks,Eastern Conference,Central,26.0,31.0,0.456140,None,2.0
8,5,Cleveland Cavaliers,Eastern Conference,Central,37.0,23.0,0.616667,None,-1.0
9,8,Detroit Pistons,Eastern Conference,Central,43.0,14.0,0.754386,None,1.0


In [ ]:
## Extract Player Stats per Game from API into a DataFrame

import time
import requests
import pandas as pd
from datetime import datetime, timedelta

BASE = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba"

def daterange(start: datetime, end: datetime):
    cur = start
    while cur <= end:
        yield cur
        cur += timedelta(days=1)

def get_game_ids_with_boxscore(start: datetime, end: datetime):
    """Return (game_id, date_yyyymmdd) for games that are likely to have boxscore data."""
    out = []
    for d in daterange(start, end):
        date_str = d.strftime("%Y%m%d")
        sb = requests.get(f"{BASE}/scoreboard", params={"dates": date_str}, timeout=30).json()

        for e in sb.get("events", []):
            # Usually "STATUS_FINAL" / "STATUS_IN_PROGRESS" / "STATUS_SCHEDULED" etc.
            status_name = e.get("status", {}).get("type", {}).get("name", "")
            # Boxscores are most reliable when Final
            if "FINAL" in status_name:
                out.append((e["id"], date_str))

        time.sleep(0.2)
    return out

def extract_player_game_stats(game_ids):
    rows = []
    for game_id in game_ids:
        data = requests.get(f"{BASE}/summary", params={"event": game_id}, timeout=30).json()

        box = data.get("boxscore", {})
        players = box.get("players", [])
        if not players:
            # No boxscore (yet) or different payload shape
            continue

        for team_block in players:
            team_id = team_block.get("team", {}).get("id")
            for category in team_block.get("statistics", []):
                stat_names = category.get("names", [])
                for a in category.get("athletes", []):
                    athlete_info = a.get("athlete", {})
                    stat_vals = a.get("stats", [])
                    stat_map = dict(zip(stat_names, stat_vals))

                    rows.append({
                        "game_id": game_id,
                        "team_id": team_id,
                        "player_id": athlete_info.get("id"),
                        "player_name": athlete_info.get("displayName"),
                        "stat_group": category.get("name"),  # e.g., "starters", "bench" (varies)
                        **stat_map
                    })

        time.sleep(0.2)

    return pd.DataFrame(rows)

# ---- Example: Start of season until yesterday ----
season_year = 2025
end = datetime.now() - timedelta(days=1)          # yesterday
start = datetime(season_year, 10, 15)

final_games = get_game_ids_with_boxscore(start, end)
print("final_games_found:", len(final_games))

game_ids = [gid for gid, _ in final_games]
df_player_game_stats = extract_player_game_stats(game_ids)

print("rows:", len(df_player_game_stats)) # 23,000+ rows of data
print(df_player_game_stats.head())

final_games_found: 897
rows: 23532
     game_id team_id player_id               player_name stat_group MIN PTS  \
0  401812722      29   4277961         Jaren Jackson Jr.       None  22  17   
1  401812722      29   5112087              Jaylen Wells       None  25  12   
2  401812722      29   3146557              Jock Landale       None  22  12   
3  401812722      29   2581018  Kentavious Caldwell-Pope       None  26  15   
4  401812722      29   4781746               Javon Small       None  28  18   

     FG  3PT   FT REB AST TO STL BLK OREB DREB PF  +/-  
0  7-20  3-9  0-0   2   1  1   0   1    0    2  2  -18  
1   5-9  2-5  0-1   1   2  2   0   0    0    1  3  -20  
2   4-6  4-5  0-0   3   0  1   0   1    1    2  3  -10  
3  5-10  2-5  3-3   2   3  1   2   0    0    2  1   -6  
4   6-9  5-6  1-2   4   8  4   0   1    2    2  2  -12  


In [ ]:
## Beta extract active players
# import requests
# import pandas as pd
# import time

# TEAMS_URL = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams"
# ROSTER_URL = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams/{}/roster"

# # Step 1: Get teams
# teams_json = requests.get(TEAMS_URL).json()

# team_ids = [
#     team_obj["team"]["id"]
#     for team_obj in teams_json["sports"][0]["leagues"][0]["teams"]
# ]

# players = []

# # Step 2: Loop through rosters
# for team_id in team_ids:
#     roster_json = requests.get(ROSTER_URL.format(team_id)).json()
    
#     for athlete in roster_json["athletes"]:
#         players.append({
#             "player_id": athlete["id"],
#             "full_name": athlete["fullName"],
#             "position": athlete.get("position", {}).get("abbreviation"),
#             "jersey": athlete.get("jersey"),
#             "team_id": team_id,
#             "active": athlete.get("active", True)
#         })
    
#     time.sleep(0.3)  # Be polite to the API

# df_players = pd.DataFrame(players)

# print(df_players.head())
# print("Total Active Players:", len(df_players))

  player_id                 full_name position jersey team_id  active
0   4278039  Nickeil Alexander-Walker        G      7       1    True
1   4869342             Dyson Daniels        G      5       1    True
2   4431941               RayJ Dennis        G      0       1    True
3   4712863            Mouhamed Gueye        F     18       1    True
4   2990984               Buddy Hield        G      8       1    True
Total Active Players: 528


In [1]:
## Extract Games for this season and add to a DataFrame

import requests
import pandas as pd
from datetime import datetime, timedelta
import time

BASE = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba"

def daterange(start_date, end_date):
    for n in range((end_date - start_date).days + 1):
        yield start_date + timedelta(n)

def extract_games(start_date, end_date):
    games = []

    for single_date in daterange(start_date, end_date):
        date_str = single_date.strftime("%Y%m%d")
        url = f"{BASE}/scoreboard?dates={date_str}"
        
        data = requests.get(url).json()

        for event in data.get("events", []):
            competition = event["competitions"][0]
            competitors = competition["competitors"]

            home = next(c for c in competitors if c["homeAway"] == "home")
            away = next(c for c in competitors if c["homeAway"] == "away")

            games.append({
                "game_id": event["id"],
                "date": event["date"],
                "season_year": event["season"]["year"],
                "season_type": event["season"]["type"],  # 2 = regular season
                "status": event["status"]["type"]["description"],
                "home_team_id": home["team"]["id"],
                "home_team_name": home["team"]["displayName"],
                "home_score": home.get("score"),
                "away_team_id": away["team"]["id"],
                "away_team_name": away["team"]["displayName"],
                "away_score": away.get("score"),
                "venue": competition.get("venue", {}).get("fullName")
            })

        time.sleep(0.2)

    return pd.DataFrame(games)

season_start = datetime(2025, 10, 15)  # adjust if needed
today = datetime.today()

df_games = extract_games(season_start, today)

print(df_games.head())
print("Total games:", len(df_games))

     game_id               date  season_year  season_type status home_team_id  \
0  401812722  2025-10-15T23:00Z         2026            1  Final           30   
1  401812723  2025-10-15T23:30Z         2026            1  Final            2   
2  401812725  2025-10-16T02:00Z         2026            1  Final           23   
3  401812724  2025-10-16T02:30Z         2026            1  Final           13   
4  401812726  2025-10-16T23:00Z         2026            1  Final           19   

       home_team_name home_score away_team_id        away_team_name  \
0   Charlotte Hornets        145           29     Memphis Grizzlies   
1      Boston Celtics        110           28       Toronto Raptors   
2    Sacramento Kings         91           12           LA Clippers   
3  Los Angeles Lakers         94            6      Dallas Mavericks   
4       Orlando Magic        132            3  New Orleans Pelicans   

  away_score                   venue  
0        116  First Horizon Coliseum  
1       